In [1]:
!pip install qwen_vl_utils

In [4]:
from google.colab import userdata
from huggingface_hub import login
import zipfile
import torch
import PIL.Image
from transformers import AutoModelForImageTextToText, AutoProcessor
from qwen_vl_utils import process_vision_info
import pandas as pd
import os

In [5]:
# hugging face login
userdata.get('HF_TOKEN')
login()

In [6]:
# unpack data
with zipfile.ZipFile('emotic.zip', 'r') as zip_ref:
    zip_ref.extractall('emotic')

In [7]:
# generic class to instatiate and prompt any of the three models
class Model:

    MODEL_IDS = {
        "qwen2.5": "Qwen/Qwen2.5-VL-3B-Instruct",
        "qwen2":   "Qwen/Qwen2-VL-7B-Instruct",
        "llama":   "meta-llama/Llama-3.2-11B-Vision-Instruct",
    }

    def __init__(self, model_name: str):

        self.model_name = model_name
        model_id = self.MODEL_IDS[model_name]

        # build forced-choice prompt
        self.labels = [
            "Peace", "Affection", "Esteem", "Anticipation", "Engagement", "Confidence",
            "Happiness", "Pleasure", "Excitement", "Surprise", "Sympathy", "Doubt/Confusion",
            "Disconnection", "Fatigue", "Embarrassment", "Yearning", "Disapproval", "Aversion",
            "Annoyance", "Anger", "Sensitivity", "Sadness", "Disquietment", "Fear", "Pain",
            "Suffering",
        ]
        self.prompt = (
            "Choose exactly ONE label from this list:\n"
            + ", ".join(self.labels)
            + "\nRespond with ONLY the label. No punctuation. No explanation."
        )

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

        # instantiate model and processor
        self.model = AutoModelForImageTextToText.from_pretrained(
            model_id,
            dtype=dtype,
            device_map="auto",
            low_cpu_mem_usage=True,
        )
        self.model.eval()

        self.processor = AutoProcessor.from_pretrained(model_id)

    def label_img(self, img: PIL.Image.Image) -> str:
        '''Return a single emotion label for the given PIL image'''
        if self.model_name in ("qwen2.5", "qwen2"):
            return self._label_qwen(img)
        else:
            return self._label_llama(img)

    def _label_qwen(self, img: PIL.Image.Image) -> str:
        '''Internal labeler for Qwen models'''
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": self.prompt},
                ],
            }
        ]

        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)

        inputs = self.processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to(self.device)

        with torch.no_grad():
            generated_ids = self.model.generate(**inputs, max_new_tokens=16)

        trimmed = [
            out[len(inp):]
            for inp, out in zip(inputs.input_ids, generated_ids)
        ]
        response = self.processor.batch_decode(
            trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]

        return self._clean(response)

    def _label_llama(self, img: PIL.Image.Image) -> str:
        '''Internal labeler for Llama'''
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": self.prompt},
                ],
            }
        ]

        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        inputs = self.processor(
            text=text,
            images=[img],
            return_tensors="pt",
        ).to(self.device)

        with torch.no_grad():
            generated_ids = self.model.generate(**inputs, max_new_tokens=16)

        trimmed = [
            out[len(inp):]
            for inp, out in zip(inputs.input_ids, generated_ids)
        ]
        response = self.processor.batch_decode(
            trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]

        return self._clean(response)

    def _clean(self, text: str) -> str:
        '''Strip whitespace/punctuation and validate against known labels'''
        cleaned = text.strip().strip(".,!?\"'").strip()
        lower = cleaned.lower()
        for label in self.labels:
            if label.lower() == lower:
                return label
        return lower

In [8]:
df = pd.read_csv("emotic/emotic/annotations.csv")
df.shape

(1000, 9)

In [9]:
# remove any entries for extra files
df['fullpath'] = df.image_path.apply(lambda x: f"emotic/emotic/images/{x}")
valid = df[df.fullpath.apply(os.path.exists)]
valid.shape

(1000, 10)

In [12]:
def process_row(row):
    '''Given row of annotations dataframe, return labels for full image and cropped image'''
    img = PIL.Image.open(row.fullpath)
    crop = img.crop((row.bbox_x1, row.bbox_y1, row.bbox_x2, row.bbox_y2)) # crop using ground truth box
    return pd.Series({
        "img_label": llm.label_img(img),
        "crop_label": llm.label_img(crop),
    })

In [ ]:
for m in ("qwen2.5", "qwen2", "llama"):
    print(m)
    llm = Model(m)
    valid[["img_label", "crop_label"]] = valid.apply(process_row, axis=1)
    valid.to_csv(f"{m}_labeled.csv")

qwen2.5


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]